# Battery Model Dispatch Plot

This notebook loads the electricity price data and plots battery charge/discharge power together with the electricity price. Set `METHOD` to `"rl"`, `"pyomo"`, or `"auto"` to choose the main dispatch source. With `SHOW_MATH_OPTIMIZATION = True`, the mathematical optimization dispatch is added to the same plot for comparison.

In [11]:
import os
import sys
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"   # must be before pyplot import

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() and (candidate / "Battery_model").exists():
            return candidate
    raise FileNotFoundError("Could not find the THESIS repository root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PRICE_FILE = ROOT / "Battery_model" / "Optimization_Pyomo" / "input_data" / "electricity_price.txt"
RL_MODEL_PATH = ROOT / "src" / "battery_model" / "battery_sac_model"
OUTPUT_FIGURE = ROOT / "notebooks" / "battery_model_dispatch_plot.png"

# Choose: "auto", "rl", "pyomo", or "heuristic".
METHOD = "auto"
SHOW_MATH_OPTIMIZATION = True

# Use one day for the dispatch comparison plot.
WINDOW_START = 120
WINDOW_HOURS = 24

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

ROOT


WindowsPath('c:/Users/thap_as/Documents/THESIS')

In [12]:
price_df = pd.read_csv(
    PRICE_FILE,
    sep=r"\s+",
    header=None,
    names=["timestamp_s", "price_eur_per_mwh"],
)

prices_full = price_df["price_eur_per_mwh"].astype(float).to_numpy()
GLOBAL_PRICE_SCALE = np.percentile(np.abs(prices_full), 95) + 1e-6

window_end = min(WINDOW_START + WINDOW_HOURS, len(prices_full))
prices_window = prices_full[WINDOW_START:window_end]

print(f"Loaded {len(prices_full)} hourly prices from {PRICE_FILE.relative_to(ROOT)}")
print(f"Using hours {WINDOW_START} to {window_end - 1} ({len(prices_window)} hours)")
print(f"Global price scale: {GLOBAL_PRICE_SCALE:.4f}")

price_df.iloc[WINDOW_START:window_end].head()


Loaded 8784 hourly prices from Battery_model\Optimization_Pyomo\input_data\electricity_price.txt
Using hours 120 to 143 (24 hours)
Global price scale: 65.5685


,timestamp_s,price_eur_per_mwh
120,432000,14.88
121,435600,11.72
122,439200,6.29
123,442800,10.19
124,446400,10.98


In [13]:
def build_dispatch_frame(method, prices, power_w, soc, power_actual_w=None, control=None, rewards=None):
    prices = np.asarray(prices, dtype=float)
    power_w = np.asarray(power_w, dtype=float)
    soc = np.asarray(soc, dtype=float)
    power_kw = power_w / 1000.0

    data = {
        "hour": np.arange(len(prices)),
        "price_eur_per_mwh": prices,
        "power_kw": power_kw,
        "charge_kw": np.clip(power_kw, 0.0, None),
        "discharge_kw": np.clip(-power_kw, 0.0, None),
        "soc": soc,
        "method": method,
    }
    if power_actual_w is not None:
        data["power_actual_kw"] = np.asarray(power_actual_w, dtype=float) / 1000.0
    if control is not None:
        data["control_u"] = np.asarray(control, dtype=float)
    if rewards is not None:
        data["reward"] = np.asarray(rewards, dtype=float)

    return pd.DataFrame(data)


def run_rl_dispatch(prices, price_scale):
    from stable_baselines3 import SAC

    from src.battery_model.evaluate import evaluate_model

    if not RL_MODEL_PATH.with_suffix(".zip").exists():
        raise FileNotFoundError(f"Missing saved RL model: {RL_MODEL_PATH.with_suffix('.zip')}")

    model = SAC.load(str(RL_MODEL_PATH))
    p_cmds, power_w, power_actual_w, soc, rewards, prices_used = evaluate_model(
        model,
        prices,
        price_scale,
    )
    return build_dispatch_frame(
        "RL SAC",
        prices_used,
        power_w,
        soc,
        power_actual_w=power_actual_w,
        control=p_cmds / 20000.0,
        rewards=rewards,
    )


def run_pyomo_dispatch(prices):
    from Battery_model.Optimization_Pyomo.model_pyomo import optimize_battery_pyomo

    result = optimize_battery_pyomo(prices, verbose=False)
    if result.get("status") != "success":
        raise RuntimeError(result.get("error", "Pyomo optimization failed."))

    return build_dispatch_frame(
        "Mathematical optimization",
        prices,
        result["P"],
        result["SOC"],
        power_actual_w=result.get("P_actual"),
        control=result.get("u"),
    )


def run_threshold_fallback(prices, p_max=20000.0, eta=0.9, e_max=2e5 * 3600.0, soc_0=0.5):
    low_price, high_price = np.quantile(prices, [0.30, 0.70])
    soc = float(soc_0)
    dt = 3600.0
    powers = []
    socs = []

    for price in prices:
        if price <= low_price:
            p_cmd = p_max
        elif price >= high_price:
            p_cmd = -p_max
        else:
            p_cmd = 0.0

        e_room_charge = (1.0 - soc) * e_max
        e_room_discharge = soc * e_max
        max_charge_p = min(p_max, (e_room_charge / dt) / max(eta, 1e-6))
        max_discharge_p = min(p_max, (e_room_discharge / dt) / max(eta, 1e-6))
        power = float(np.clip(p_cmd, -max_discharge_p, max_charge_p))
        soc = float(np.clip(soc + eta * power * dt / e_max, 0.0, 1.0))

        powers.append(power)
        socs.append(soc)

    return build_dispatch_frame("Threshold fallback", prices, powers, socs)


def dispatch_from_method(method, prices, price_scale):
    method = method.lower()
    if method not in {"auto", "rl", "pyomo", "heuristic"}:
        raise ValueError('METHOD must be one of: "auto", "rl", "pyomo", "heuristic".')

    candidates = ["rl", "pyomo", "heuristic"] if method == "auto" else [method]
    notes = []
    for candidate in candidates:
        try:
            if candidate == "rl":
                return run_rl_dispatch(prices, price_scale), notes
            if candidate == "pyomo":
                return run_pyomo_dispatch(prices), notes
            return run_threshold_fallback(prices), notes
        except Exception as exc:
            notes.append(f"{candidate} unavailable: {exc}")
            if method != "auto":
                raise

    raise RuntimeError("No dispatch method could be evaluated.")


def objective_value_eur(df):
    return float(np.sum((df["price_eur_per_mwh"] / 1e6) * (df["power_kw"] * 1000.0)))


def summarize_dispatch(df):
    objective_eur = objective_value_eur(df)
    return pd.DataFrame([
        {
            "method": df["method"].iloc[0],
            "window_hours": len(df),
            "objective_value_eur": objective_eur,
            "total_cost_eur": objective_eur,
            "charge_energy_kwh": float(df["charge_kw"].sum()),
            "discharge_energy_kwh": float(df["discharge_kw"].sum()),
            "min_soc": float(df["soc"].min()),
            "max_soc": float(df["soc"].max()),
        }
    ])


In [14]:
math_dispatch_df = None

if SHOW_MATH_OPTIMIZATION:
    try:
        math_dispatch_df = run_pyomo_dispatch(prices_window)
        print("Prepared mathematical optimization dispatch for the plot.")
    except Exception as exc:
        print(f"Mathematical optimization unavailable: {exc}")

dispatch_df, selection_notes = dispatch_from_method(METHOD, prices_window, GLOBAL_PRICE_SCALE)

if dispatch_df["method"].iloc[0] == "Mathematical optimization":
    math_dispatch_df = dispatch_df

print(f"Selected method: {dispatch_df['method'].iloc[0]}")
for note in selection_notes:
    print(note)

summary_frames = [summarize_dispatch(dispatch_df)]
if math_dispatch_df is not None and math_dispatch_df is not dispatch_df:
    summary_frames.append(summarize_dispatch(math_dispatch_df))

display(pd.concat(summary_frames, ignore_index=True))
dispatch_df.head()


Prepared mathematical optimization dispatch for the plot.
Selected method: RL SAC


,method,window_hours,objective_value_eur,total_cost_eur,charge_energy_kwh,discharge_energy_kwh,min_soc,max_soc
0,RL SAC,24,-9.551140,-9.551140,111.111111,222.222222,0.000000,1.0
1,Mathematical optimization,24,-9.619101,-9.619101,129.206396,239.819346,0.002242,1.0


,hour,price_eur_per_mwh,power_kw,charge_kw,discharge_kw,soc,method,power_actual_kw,control_u,reward
0,0,14.88,19.510260,19.510260,0.0,0.587796,RL SAC,17.559234,0.975513,-0.290313
1,1,11.72,19.633555,19.633555,0.0,0.676147,RL SAC,17.670200,0.981678,-0.230105
2,2,6.29,19.665174,19.665174,0.0,0.764640,RL SAC,17.698657,0.983259,-0.123694
3,3,10.19,19.462786,19.462786,0.0,0.852223,RL SAC,17.516507,0.973139,-0.198326
4,4,10.98,19.416466,19.416466,0.0,0.939597,RL SAC,17.474819,0.970823,-0.213193


In [15]:
def plot_battery_dispatch(df, math_df=None, output_path=None):
    method = df["method"].iloc[0]
    hours = df["hour"].to_numpy()
    x_min, x_max = 0, 23
    x_ticks = np.arange(x_min, x_max + 1, 1)
    primary_label = "RL" if method == "RL SAC" else method

    fig, (ax_power, ax_soc) = plt.subplots(
        2,
        1,
        figsize=(14, 5.2),
        sharex=True,
        gridspec_kw={"height_ratios": [3.0, 1.0], "hspace": 0.08},
    )

    ax_power.fill_between(
        hours,
        0,
        df["charge_kw"],
        step="post",
        color="#7777ff",
        alpha=0.45,
        label=f"{primary_label} charge",
    )
    ax_power.fill_between(
        hours,
        0,
        df["discharge_kw"],
        step="post",
        color="#79bd7a",
        alpha=0.50,
        label=f"{primary_label} discharge",
    )
    ax_power.step(hours, df["charge_kw"], where="post", color="#3434a8", linewidth=1.1)
    ax_power.step(hours, df["discharge_kw"], where="post", color="#276b35", linewidth=1.1)

    max_power = max(float(df["charge_kw"].max()), float(df["discharge_kw"].max()), 1.0)
    if math_df is not None and math_df is not df:
        math_hours = math_df["hour"].to_numpy()
        ax_power.step(
            math_hours,
            math_df["charge_kw"],
            where="post",
            color="#0000c7",
            linewidth=2.0,
            linestyle="--",
            label="Math charge",
        )
        ax_power.step(
            math_hours,
            math_df["discharge_kw"],
            where="post",
            color="#0b5d1e",
            linewidth=2.0,
            linestyle="--",
            label="Math discharge",
        )
        max_power = max(
            max_power,
            float(math_df["charge_kw"].max()),
            float(math_df["discharge_kw"].max()),
        )
    ax_power.set_ylim(0, max_power * 1.25)
    ax_power.set_xlim(x_min, x_max)
    ax_power.set_ylabel("Power [kW]", fontweight="bold")
    ax_power.grid(True, alpha=0.28)
    ax_power.tick_params(labelbottom=False)

    ax_price = ax_power.twinx()
    ax_price.plot(
        hours,
        df["price_eur_per_mwh"],
        color="red",
        linewidth=2.0,
        label="Electricity price",
    )
    ax_price.set_ylabel("Electricity price [EUR/MWh]", fontweight="bold")

    handles_1, labels_1 = ax_power.get_legend_handles_labels()
    handles_2, labels_2 = ax_price.get_legend_handles_labels()
    ax_power.legend(
        handles_1 + handles_2,
        labels_1 + labels_2,
        loc="upper left",
        frameon=True,
        ncol=3,
    )

    objective_lines = [f"{primary_label} objective: {objective_value_eur(df):.2f} EUR"]
    if math_df is not None and math_df is not df:
        objective_lines.append(f"Math objective: {objective_value_eur(math_df):.2f} EUR")
    ax_power.text(
        0.985,
        0.06,
        "\n".join(objective_lines),
        transform=ax_power.transAxes,
        ha="right",
        va="bottom",
        fontsize=9,
        bbox={"facecolor": "white", "edgecolor": "#bbbbbb", "alpha": 0.88, "boxstyle": "round,pad=0.35"},
    )

    ax_soc.plot(
        hours,
        df["soc"] * 100.0,
        color="#1f1f1f",
        linewidth=1.9,
        label=f"{primary_label} SOC",
    )
    ax_soc.fill_between(hours, 0, df["soc"] * 100.0, color="#1f1f1f", alpha=0.08)
    if math_df is not None and math_df is not df:
        ax_soc.plot(
            math_df["hour"],
            math_df["soc"] * 100.0,
            color="#555555",
            linewidth=1.8,
            linestyle="--",
            label="Math SOC",
        )
    ax_soc.set_ylim(0, 100)
    ax_soc.set_xlim(x_min, x_max)
    ax_soc.set_xticks(x_ticks)
    ax_soc.set_xlabel("Time [h]")
    ax_soc.set_ylabel("SOC [%]", fontweight="bold")
    ax_soc.grid(True, alpha=0.28)
    ax_soc.legend(loc="upper left", frameon=True)

    if math_df is not None and math_df is not df:
        title = f"Battery dispatch: {primary_label} vs mathematical optimization | hours {WINDOW_START}-{window_end - 1}"
    else:
        title = f"Battery dispatch from {method} | hours {WINDOW_START}-{window_end - 1}"
    ax_power.set_title(title)
    fig.subplots_adjust(top=0.90)

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Saved figure to {output_path.relative_to(ROOT)}")

    return fig


try:
    math_dispatch_df
except NameError:
    math_dispatch_df = None

if SHOW_MATH_OPTIMIZATION and math_dispatch_df is None:
    try:
        math_dispatch_df = run_pyomo_dispatch(prices_window)
        print("Prepared mathematical optimization dispatch for the plot.")
    except Exception as exc:
        print(f"Mathematical optimization unavailable: {exc}")

plot_battery_dispatch(dispatch_df, math_dispatch_df, OUTPUT_FIGURE)

Saved figure to notebooks\battery_model_dispatch_plot.png


<Figure size 1680x624 with 3 Axes>

To compare RL against the mathematical optimizer, keep `METHOD = "auto"` or `METHOD = "rl"` and `SHOW_MATH_OPTIMIZATION = True`. To show only mathematical optimization as the main dispatch, set `METHOD = "pyomo"`. Pyomo requires a working IPOPT installation.